# Both models / 08 — FastAPI integration

Load the exact saved pipelines and exercise the FastAPI prediction endpoint without a running server.

Run every cell from top to bottom. Use the project `.venv` kernel in VS Code. No season field is used. Data and test scores are synthetic.

In [1]:
from pathlib import Path
import sys, json, inspect
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ml_service/workflows.py").exists())
sys.path.insert(0, str(ROOT))
from ml_service import workflows as w
print("Project:", ROOT)
print("Synthetic demonstration: real-world accuracy has not been measured.")

Project: /workspace/scratch/71e3a62599e6/YatraSense_Organic
Synthetic demonstration: real-world accuracy has not been measured.


In [2]:
from fastapi.testclient import TestClient
from ml_service.main import app
from ml_service.inference import artifacts
from ml_service.workflows import ACTIVITIES
client = TestClient(app)
assert client.get("/health").status_code == 200
catalog = artifacts()[3]
destinations = [{"id": r.destination_id, "name": r.destination_name, "area": r.area, "activities": [a for a in ACTIVITIES if r[a] == 1], "popularity_score": r.popularity_score} for _,r in catalog.iterrows()]
payload = {"travel_date": "2026-10-15", "activities": ["heritage_pilgrimage"], "crowd_preference": "Any", "area": "All", "destinations": destinations}
response = client.post("/predict", json=payload)
assert response.status_code == 200, response.text
result = response.json()
display(pd.DataFrame(result["results"])[["name","crowd","popularity","recommendation_score"]])

/root/.local/lib/python3.12/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


,name,crowd,popularity,recommendation_score
0,Patan Durbar Square,High,High,0.9101
1,Patan Museum,High,Medium,0.8972
2,Bhaktapur Durbar Square,High,High,0.8929
3,Golden Temple,High,Medium,0.8825
4,Boudhanath,High,High,0.8744
5,Kathmandu Durbar Square,High,High,0.8636
6,Pashupatinath,High,High,0.8397
7,Garden of Dreams,High,Medium,0.8379
8,World Peace Pagoda,High,High,0.8366
9,Nyatapola Temple,High,Medium,0.8135


In [3]:
payload["crowd_preference"] = "Low"
quiet = client.post("/predict", json=payload).json()
assert all(r["crowd"] == "Low" for r in quiet["results"])
payload["activities"] = ["wildlife_birdwatching"]
empty = client.post("/predict", json=payload).json()
assert empty["results"] == []
print("Crowd filters and empty activity matches verified.")

Crowd filters and empty activity matches verified.


Website integration: React → Django `/api/plan/` → FastAPI `/predict` → saved crowd pipeline and recommendation pipeline. Django enriches results with SQLite destination records, photos and maps. Saved customer plans are stored in SQLite. Start the three services using the README. Offline photos are bundled; OpenStreetMap tiles require internet.